In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Census Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying Census data
indicator_name     = df_params[df_params['Type'] == 'indicator_name' ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'       ]['Input'].values[0]
sample_type        = df_params[df_params['Type'] == 'sample'         ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'      ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'     ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'    ]['Input'].values[0]
margin_of_error    = df_params[df_params['Type'] == 'margin_of_error']['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'       ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'     ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'       ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'   ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'      ]['Input'].values[0]

# View
print(indicator_name)
print(estimate)
print(sample_type)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Margin of error: " + margin_of_error)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

# Import about table
df_about = pd.read_excel(os.path.join(path_config0, 'About Indicators.xlsx'), sheet_name = 'Indicators')
df_about = df_about[df_about['Indicator'] == indicator_name]
folder = df_about.Folder.values[0]
MOE_thresh = df_about['MOE Threshold'].values[0]
print(folder)
print('MOE threshold: ' + str(MOE_thresh) + '%')

## Import Data

In [ ]:
# Import data by geography
path_in = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder)

try: 
    df_counties = pd.read_excel(os.path.join(path_in, indicator_name + ' Counties ' + estimate + '.xlsx'), sheet_name = 'Counties')
    display(df_counties.head(3))
except Exception as e: print(e)

try: 
    df_mpo = pd.read_excel(os.path.join(path_in, indicator_name + ' MPO ' + estimate + '.xlsx'), sheet_name = 'MPO')
    display(df_mpo.head(3))
except Exception as e: print(e)

try: 
    df_msa = pd.read_excel(os.path.join(path_in, indicator_name + ' MSA ' + estimate + '.xlsx'), sheet_name = 'MSA')
    display(df_msa.head(3))
except Exception as e: print(e)

try: 
    df_counties = pd.read_excel(os.path.join(path_in, indicator_name + ' Counties ' +  re.sub('ACS', 'PUMS', estimate) + '.xlsx'), sheet_name = 'Counties')
    display(df_counties.head(3))
except Exception as e: print(e)

try: 
    df_mpo = pd.read_excel(os.path.join(path_in, indicator_name + ' MPO ' +  re.sub('ACS', 'PUMS', estimate) + '.xlsx'), sheet_name = 'MPO')
    display(df_mpo.head(3))
except Exception as e: print(e)

In [ ]:
if indicator_name == 'Cost_3':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'Vacant']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'Vacant']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'Vacant']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'Vacant']

if indicator_name == 'Cost_5':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'Owner occupied']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'Owner occupied']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'Owner occupied']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'Owner occupied']

if indicator_name == 'Health_2':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'No health insurance coverage']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'No health insurance coverage']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'No health insurance coverage']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'No health insurance coverage']

if indicator_name == 'Income_4':
    if geography == 'Tracts':
        df_tracts = df_tracts[df_tracts['Variable'] == 'Total Income in the past 12 months below poverty level']
    if geography == 'Counties':
        df_counties = df_counties[df_counties['Variable'] == 'Total Income in the past 12 months below poverty level']
        df_mpo      = df_mpo     [df_mpo     ['Variable'] == 'Total Income in the past 12 months below poverty level']
    if geography == 'MSA':
        df_msa = df_msa[df_msa['Variable'] == 'Total Income in the past 12 months below poverty level']

if indicator_name == 'Cost_6':
    # df_counties = df_counties[df_counties['housing_type'] == 'Owners and Renters']
    # df_mpo      = df_mpo     [df_mpo     ['housing_type'] == 'Owners and Renters']
    df_mpo = df_mpo[(df_mpo['housing_type'] == 'Owner') | (df_mpo['housing_type'] == 'Renter')]
    df_counties = df_counties[(df_counties['housing_type'] == 'Owner') | (df_counties['housing_type'] == 'Renter')]


In [ ]:
# remove state FIPS field
# remove "All" category for race/ethnicity
# make sure index is removed
# make sure proportions are now percentages

try:
    df_counties = df_counties.drop(['State FIPS', 'County FIPS'], axis = 1)
    df_mpo      = df_mpo     .drop(['State FIPS'               ], axis = 1)
    
    df_counties.columns = [col.lower() for col in df_counties.columns]
    df_mpo     .columns = [col.lower() for col in df_mpo     .columns]
    
    df_counties.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_counties.columns]
    df_mpo     .columns = [re.sub('[\s+]', '_', col.strip()) for col in df_mpo     .columns]
    
    df_counties.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_counties.columns]
    df_mpo     .columns = [re.sub('\\?'  , '' , col.strip()) for col in df_mpo     .columns]
except:
    pass

try:      
    df_msa  = df_msa.reset_index(drop = True)
    df_msa.columns = [x.lower() for x in df_msa.columns]
    df_msa.columns = [re.sub('[\s+]', '_', col.strip()) for col in df_msa.columns]
    df_msa.columns = [re.sub('\\?'  , '' , col.strip()) for col in df_msa.columns]
except:
    pass
    
if indicator_name == 'Cost_6':
    df_counties.loc[df_counties['housing_burden'] == 'Cost burden <=30%'        , 'housing_burden'] = 'Cost burden less than 30 perc'
    df_counties.loc[df_counties['housing_burden'] == 'Cost burden >30% to <=50%', 'housing_burden'] = 'Cost burden 30 to 50 perc'
    df_counties.loc[df_counties['housing_burden'] == 'Cost burden >50%'         , 'housing_burden'] = 'Cost burden greater than 50 perc'
    
    df_mpo.loc[df_mpo['housing_burden'] == 'Cost burden <=30%'        , 'housing_burden'] = 'Cost burden less than 30 perc'
    df_mpo.loc[df_mpo['housing_burden'] == 'Cost burden >30% to <=50%', 'housing_burden'] = 'Cost burden 30 to 50 perc'
    df_mpo.loc[df_mpo['housing_burden'] == 'Cost burden >50%'         , 'housing_burden'] = 'Cost burden greater than 50 perc'

    

In [ ]:
try:
    print('Columns: ' + str(list(df_mpo.columns)))
    print('Variables: '      + str(unique(df_mpo.variable      .values)))
    print('Race/Ethnicity: ' + str(unique(df_mpo.race_ethnicity.values)))
    display(df_counties.head(3), df_mpo.head(3))
except:
    pass

try:
    print('Columns: ' + str(list(df_msa.columns)))
    print('Variables: '      + str(unique(df_msa.variable      .values)))
    print('Race/Ethnicity: ' + str(unique(df_msa.race_ethnicity.values)))
    display(df_msa.head(3))
except:
    pass

try:
    print('Columns: ' + str(list(df_mpo.columns)))
    print('Variables: '      + str(unique(df_mpo.income_bracket.values)))
    display(df_counties.head(3), df_mpo.head(3))
except:
    pass

## Data Visualization

#### Line Graphs

In [ ]:
path_plots = os.path.join(path_main, report_theme, sp_folder_out, indicator_name + ' ' + folder, 'plots')
# df_plot = df_counties.copy()
# df_plot = df_mpo.copy()
df_plot = df_msa.copy()


# Plotting setup
by_race = False
race_ethnicity = 'race_ethnicity'
loop_vars = False
variable = 'variable'
x = 'year'
y = 'percentage'
color = 'msa'
line_dash = None
markers = True
plot_title = folder + '_' + estimate + '_Peer MSA'
plot_name  = folder + '_' + estimate + '_Peer MSA'
# plot_title = 'Percent of regional median household income' + estimate + ' SACOG MPO'
# plot_name  = 'Percent of regional median household income' + estimate + ' SACOG MPO'
export = True

In [ ]:
def plot_lines(
    df=df_plot
     , loop_vars=loop_vars, by_race=by_race, race_ethnicity=race_ethnicity, variable=variable
     , x=x, y=y
     , color=color, line_dash=line_dash, markers=markers
     , plot_title=plot_title, plot_name=plot_name
     , export=export
):
    if race_ethnicity != None:
        if by_race:
            df = df[df[race_ethnicity] != 'All']
        else:
            df = df[df[race_ethnicity] == 'All'].drop(race_ethnicity, axis = 1)  

    if loop_vars:
        vars = unique(df[variable].values)
        for var in vars:
            df2 = df[df[variable] == var]
            fig = px.line(df2, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
            fig.update_layout(title = plot_title + ' - ' + str(var))
            if export:
                fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', var + '_', 'line.html'])))
                
            fig.update_layout(autosize=False, width=1050, height=450)
            
    else:
        fig = px.line(df, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
        fig.update_layout(title = plot_title)
        if export:
            fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', 'line.html'])))

        fig.update_layout(autosize=False, width=1050, height=450)

    return fig.show()
        
plot_lines()

#### Bar Plots

- Does it make sense to do it just for the most recent year?  Over 5 year periods?  Whole time-series?
- Side by side bar plots vs stacked?
- Facet_wrap could be an option too?

#### Any other ideas for data visualization are welcome

- No pie charts

#### Code graveyard

In [ ]:
# path_plots = os.path.join(path_main, report_theme, sp_folder_out, indicator_name, 'plots')
# df = df_acs.copy()

# if by_race == True:
#     df = df[df[race_ethnicity] != 'All']
# else:
#     df = df[df[race_ethnicity] == 'All'].drop('race_ethnicity', axis = 1)
# if by_vars == True:
#     vars = unique(df[variable].values)
#     for var in vars:
#         df2 = df[df[variable] == var]
#         fig = px.line(df2, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
#         fig.update_layout(title = plot_title)
#         # fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', var + '_', 'line_.html'])))
        
# else:
#     fig = px.line(df, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
#     fig.update_layout(title = plot_title)
#     # fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', 'line_.html'])))

# fig.show()

In [ ]:
# path_plots = os.path.join(path_main, report_theme, sp_folder_out, indicator_name, 'plots')

# def plot_lines(data, indicator_name, estimate, x, y, group, variables):

#     df_plot = data.copy()
#     color  = group
#     labels = group
#     vars   = unique(variables)
#     x = x
#     y = y
#     vars = unique(df_plot[variables].values)

#     if group == 'race_ethnicity':
#         df_plot = df_plot[df_plot['race_ethnicity'] != 'All']
#         df_plot = df_plot[['year', 'variable', 'race_ethnicity', 'percentage']]
#         df_plot = df_plot[df_plot['race_ethnicity'] != 'All']

#     if group in ['MSA', 'County']:
#         if group == 'County':
#             df_plot = df_plot[['year', 'county_name', 'variable', 'race_ethnicity', 'percentage']]
#         if group == 'MSA':
#             df_plot = df_plot[['year', 'msa', 'variable', 'race_ethnicity', 'percentage']]
#         df_plot = df_plot[df_plot['race_ethnicity'] == 'All'].drop('race_ethnicity', axis = 1)       


#     for var in vars:
#         df_plot2 = df_plot[df_plot['variable'] == var]
        
#         fig = px.line(df_plot2
#                          , x = x
#                          , y = y
#                          , color = color
#                          , markers = True
#                          , labels = group
#                         )
            
#         fig.update_layout(title = var + ' by ' + group)
        
#         fig.write_html(
#             os.path.join(
#                 path_plots
#                 , ''.join([indicator_name + '_'
#                            , var  + '_'
#                            , group + '_'
#                            , estimate  + '_'
#                            , 'line_'
#                            , '.html'])
#             )
#         )
    

In [ ]:
# plot_lines(data = df_acs
#            , indicator_name = indicator_name
#            , estimate = estimate
#            , x = 'year'
#            , y = 'percentage'
#            , group = 'county_name'
#            , variables = 'variable')